# Quantifying Mtb on a single-cell level across large tissue slices

First work on a single example to determine thresholds then iterate over all slices.

In [1]:
import os
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

# Load previously calculated results

In [2]:
df = pd.read_pickle('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/sc_mtb_tissue.pkl')
df

,zarr_address,region_id,mean_intensity_ch1,mean_intensity_ch2,area,bbox_rmin,bbox_rmax,bbox_cmin,bbox_cmax,mouse_N,rep_N,split,condition,timer_ratio,x,y
0,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...,1,669.458333,1154.916667,72.0,12,28,24285,24311,2,1,None,DMSO,0.579659,20,24298
1,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...,2,394.212121,602.606061,33.0,31,46,25260,25278,2,1,None,DMSO,0.654179,38,25269
2,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...,3,613.163636,943.381818,55.0,43,59,25134,25166,2,1,None,DMSO,0.649963,51,25150
3,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...,4,737.836957,1462.576087,92.0,46,69,24763,24787,2,1,None,DMSO,0.504478,57,24775
4,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...,5,257.228070,396.228070,57.0,47,64,27340,27364,2,1,None,DMSO,0.649192,55,27352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
452360,rep2_mouse16_top.zarr,501,3168.583333,805.000000,156.0,34194,34228,2815,2845,16,2,top,PZA,3.936128,34211,2830
452361,rep2_mouse16_top.zarr,502,3707.631579,913.567251,171.0,34204,34231,2751,2785,16,2,top,PZA,4.058411,34217,2768
452362,rep2_mouse16_top.zarr,503,4872.142857,1180.142857,7.0,34211,34224,18594,18607,16,2,top,PZA,4.128435,34217,18600
452363,rep2_mouse16_top.zarr,504,3681.236842,837.684211,38.0,34244,34263,765,781,16,2,top,PZA,4.394540,34253,773


In [3]:
# Define the root directory to search in
root_dir = Path('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice')
dapi_seg_list=[]
for zarr_address in tqdm(df['zarr_address'].unique()):
    
    # 1. Find the full path to the Zarr file
    # We search specifically for this filename pattern within the root structure
    found_zarrs = list(root_dir.glob(f"rep*/mouse_*/zarr/{zarr_address}"))
    
    if not found_zarrs:
        print(f"⚠️ Zarr file not found on disk: {zarr_address}")
        continue
        
    zarr_path = found_zarrs[0]
    
    # 2. Check for dapi_segmentation inside the Zarr folder
    # Note: In OME-Zarr, 'labels' is a subdirectory
    # We look for any folder/file starting with 'dapi_segmentation' inside 'labels'
    dapi_seg_paths = list(zarr_path.glob("labels/dapi_segmentation*"))
    
    if dapi_seg_paths:
        print(f"✅ Found segmentation: {dapi_seg_paths[0]}")
        dapi_seg_list.append(dapi_seg_paths[0])
    else:
        print(f"❌ MISSING segmentation for: {zarr_address}")

  0%|          | 0/54 [00:00<?, ?it/s]

❌ MISSING segmentation for: 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551.zarr
❌ MISSING segmentation for: 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5552.zarr
❌ MISSING segmentation for: 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5553.zarr
❌ MISSING segmentation for: 20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5555.zarr
❌ MISSING segmentation for: 20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375.zarr
✅ Found segmentation: /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574.zarr/labels/dapi_segmentation
✅ Found segmentation: /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_2/zarr/20

In [4]:
seg_path = dapi_seg_list[0]

In [5]:
# -- B. Load the Zarr Array --
try:
    # Open the Zarr array (read-only mode is safer)
    # We access the '0' scale level (highest resolution) by default in OME-Zarr
    # Adjust ['0'] if your segmentation isn't multiscaled
    z_grp = zarr.open(str(seg_path), mode='r')
    
    # OME-Zarr often stores data in a numbered subgroup like '0', '1', '2' (pyramid levels)
    # We try to grab the high-res level '0'
    if '0' in z_grp:
        dapi_mask = z_grp['0']#[:]
    else:
        # If it's a flat array (not OME-Zarr pyramid), just read it directly
        dapi_mask = z_grp#[:]
        
    # Squeeze to remove extra dimensions (e.g., T or C if they exist and are size 1)
    # e.g., (1, 1, Y, X) -> (Y, X)
    # dapi_mask = np.squeeze(dapi_mask)
    
except Exception as e:
    print(f"Error reading {seg_path}: {e}")
    # continue


Error reading /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574.zarr/labels/dapi_segmentation: name 'zarr' is not defined


In [ ]:
import gc
import re

import numpy as np
import pandas as pd
import zarr
from skimage.measure import regionprops_table
from tqdm import tqdm

# --- 1. Prepare Metadata Lookup ---
if 'condition' in df.columns:
    meta_lookup = df[['rep_N', 'mouse_N', 'condition']].drop_duplicates().set_index(['rep_N', 'mouse_N'])['condition'].to_dict()
else:
    print("⚠️ Warning: 'condition' column missing from df. Conditions will be NaN.")
    meta_lookup = {}

# Define output directory for the individual pickles
output_dir = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/dapi_pickles/'
os.makedirs(output_dir, exist_ok=True)

# --- 2. Iterate and Process (RAM SAFE) ---
print(f"Extracting nuclei coordinates from {len(dapi_seg_list)} images...")

for seg_path in tqdm(dapi_seg_list):
    
    # -- A. Extract Metadata from Path --
    match = re.search(r'/rep(\d+)/mouse_?(\d+)/', str(seg_path))
    
    if match:
        rep = int(match.group(1))
        mouse = int(match.group(2))
        condition = meta_lookup.get((rep, mouse), "Unknown")
    else:
        continue

    # -- B. Load the Zarr Array --
    try:
        z_grp = zarr.open(str(seg_path), mode='r')
        if '0' in z_grp:
            dapi_mask = z_grp['0'][:]
        else:
            dapi_mask = z_grp[:]
        
        dapi_mask = np.squeeze(dapi_mask)
        
    except Exception as e:
        print(f"Error reading {seg_path}: {e}")
        continue

    # -- C. Extract Centroids --
    try:
        props = regionprops_table(dapi_mask, properties=('label', 'centroid'))
        
        # Create a temporary DataFrame
        temp_df = pd.DataFrame(props)
        
        # Free up the mask memory immediately
        del dapi_mask
        
        if 'centroid-0' in temp_df.columns and 'centroid-1' in temp_df.columns:
            temp_df = temp_df.rename(columns={'centroid-0': 'y', 'centroid-1': 'x'})
            
            # Add metadata columns
            temp_df['rep_N'] = rep
            temp_df['mouse_N'] = mouse
            temp_df['condition'] = condition
            
            # Extract Zarr Filename for unique saving
            parent_zarr = pd.Series(str(seg_path)).str.extract(r'([^/]+\.zarr)')[0][0]
            temp_df['zarr_address'] = parent_zarr
            
            # -- SAVE IMMEDIATELY --
            # Use zarr name to ensure uniqueness (avoids overwrites)
            save_path = os.path.join(output_dir, f"{parent_zarr}_nuclei.pkl")
            temp_df.to_pickle(save_path)
            
            # -- CLEANUP --
            del temp_df
            del props
            gc.collect() # Force Python to release memory
            
    except ValueError:
        continue
    except Exception as e:
        print(f"Error processing {seg_path}: {e}")
        continue

print("Done! All files saved individually.")

Extracting nuclei coordinates from 16 images...


  0%|                                                                                           | 0/16 [00:00<?, ?it/s]

In [ ]:
yabbadabbado

In [ ]:
import os
import re

import numpy as np
import pandas as pd
import zarr

# from tqdm import tqdm
from skimage.measure import regionprops_table

# --- 1. Prepare Metadata Lookup ---
# Create a dictionary to quickly find 'condition' based on (rep, mouse)
# This prevents us from having to do slow merges later.
# Format: { (1, 2): 'H2O', (1, 3): 'DMSO', ... }
if 'condition' in df.columns:
    meta_lookup = df[['rep_N', 'mouse_N', 'condition']].drop_duplicates().set_index(['rep_N', 'mouse_N'])['condition'].to_dict()
else:
    print("⚠️ Warning: 'condition' column missing from df. Conditions will be NaN.")
    meta_lookup = {}

nuclei_data_list = []

# --- 2. Iterate and Process ---
print(f"Extracting nuclei coordinates from {len(dapi_seg_list)} images...")

for seg_path in tqdm(dapi_seg_list):
    
    # -- A. Extract Metadata from Path --
    # We use the full path string to find Rep and Mouse numbers
    # Regex looks for /repX/ and /mouse_Y/
    match = re.search(r'/rep(\d+)/mouse_?(\d+)/', str(seg_path))
    
    if match:
        rep = int(match.group(1))
        mouse = int(match.group(2))
        condition = meta_lookup.get((rep, mouse), "Unknown")
    else:
        print(f"Skipping (Metadata not found in path): {seg_path}")
        continue

    # -- B. Load the Zarr Array --
    try:
        # Open the Zarr array (read-only mode is safer)
        # We access the '0' scale level (highest resolution) by default in OME-Zarr
        # Adjust ['0'] if your segmentation isn't multiscaled
        z_grp = zarr.open(str(seg_path), mode='r')
        
        # OME-Zarr often stores data in a numbered subgroup like '0', '1', '2' (pyramid levels)
        # We try to grab the high-res level '0'
        if '0' in z_grp:
            dapi_mask = z_grp['0'][:]
        else:
            # If it's a flat array (not OME-Zarr pyramid), just read it directly
            dapi_mask = z_grp[:]
            
        # Squeeze to remove extra dimensions (e.g., T or C if they exist and are size 1)
        # e.g., (1, 1, Y, X) -> (Y, X)
        dapi_mask = np.squeeze(dapi_mask)
        
    except Exception as e:
        print(f"Error reading {seg_path}: {e}")
        continue

    # -- C. Extract Centroids --
    # regionprops_table is much faster than looping through regions
    try:
        props = regionprops_table(dapi_mask, properties=('label', 'centroid'))
        
        # Create a temporary DataFrame
        temp_df = pd.DataFrame(props)
        
        # Rename columns standard names (centroid-0 is Y, centroid-1 is X)
        # Note: If 3D, you might get centroid-0 (Z), -1 (Y), -2 (X). Check shape!
        if 'centroid-0' in temp_df.columns and 'centroid-1' in temp_df.columns:
            temp_df = temp_df.rename(columns={'centroid-0': 'y', 'centroid-1': 'x'})
            
            # Add metadata columns
            temp_df['rep_N'] = rep
            temp_df['mouse_N'] = mouse
            temp_df['condition'] = condition
            
            # Store the basename of the Zarr for reference
            # Assuming seg_path is .../filename.zarr/labels/dapi_segmentation
            # We go up 3 levels to find the .zarr folder name
            parent_zarr = pd.Series(str(seg_path)).str.extract(r'([^/]+\.zarr)')[0][0]
            temp_df['zarr_address'] = parent_zarr
            
            nuclei_data_list.append(temp_df)
        temp_df.to_pickle(f'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/dapi_coords_rep{rep}_mouse{mouse}.pkl')
    except ValueError:
        # Happens if mask is empty
        continue

# --- 3. Combine into Final DataFrame ---
if nuclei_data_list:
    nuclei_df = pd.concat(nuclei_data_list, ignore_index=True)
    print("Done!")
    print(nuclei_df.head())
else:
    print("No nuclei data extracted.")